# Lab 12 starter: value-based segments

This notebook is standalone. In the recap cell, the snapshot is loaded and the
chapter's four behavioral segments are rebuilt, so that you can compare your
segments with them.

The lab prompt is at the end of Chapter 12, under **Exercises**.

**Build lab.** Cluster the customers on `revenue_12m`, `orders_12m`, and
`tenure_months` only, after scaling them, with k = 4. With four groups, the value
segments can be compared with the four behavioral segments. Report the inertia and
the silhouette score for k from 2 to 7 for reference. Profile the four value segments
in the original units, and state for each whether its mean revenue, orders, and
tenure are above or below the overall means. Compare the value segments with the
behavioral segments in a crosstab. For each behavioral segment, report the value
segment that holds most of its customers, and the share it holds.

**Evaluate lab.** Evaluate the value segments with the chapter's three checks:
the silhouette score on the three scaled value features beside that of the business
types, the ARI across ten 80% subsamples, and the churn rate of each value segment
beside the behavioral segments. Compute the spread in churn rates of each grouping,
the highest rate minus the lowest, in percentage points. As in the chapter's
usefulness check, the grouping with the larger spread is given to the retention
team. Name that grouping.

## Running this notebook

Run the setup cell below first, then the recap cell, then work down. In Google
Colab nothing needs to be installed beyond that cell. Locally, use the
`pyba-core` environment from Appendix A.

You will enter your work in the cells marked `# TODO`. Everything else is
provided. When you are unsure what a step produced, print it: `.head()` for a
table or a column, and the bare name for a number.

**If you are in Google Colab:** this notebook opened read-only from GitHub. Click
**Copy to Drive** in the toolbar (or *File > Save a copy in Drive*) before you
edit anything, then work in that copy. It saves to your Google Drive under
*Colab Notebooks*. Edits made to the read-only original are lost when the tab
closes.

In [ ]:
# Setup. Run this cell once per session. It installs this lab's packages;
# on Google Colab it also fetches the course data.
%pip install -q pandas scikit-learn
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Recap: the Chapter 12 segments

The recap cell contains the chapter's code. `snap` is the customer snapshot, with
872 rows. `X` is the table of the six behavior features after scaling. `km` is the
chapter's k-means model with k = 4, and the columns `segment` and `segment_name`
hold each customer's behavioral segment. The cell prints the number of customers in
each segment, so that you can compare your segments with them in Step 6.

In [ ]:
# --- Chapter 12 recap: run this cell first ---
import pandas as pd

from pyba import DATA_DIR
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

snap = pd.read_csv(DATA_DIR / "pw_customer_snapshot.csv")
features = ["orders_12m", "avg_order_value", "pct_online",
            "avg_discount_pct", "aug_jan_share", "weeks_since_last_order"]

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(snap[features]), columns=features)

km = KMeans(n_clusters=4, n_init=10, random_state=0)
snap["segment"] = km.fit_predict(X)
names = {0: "Seasonal institutions", 1: "Mid-range accounts",
         2: "Frequent buyers", 3: "Small online buyers"}
snap["segment_name"] = snap["segment"].map(names)

snap["segment_name"].value_counts()

## Step 1: Scale the three value features

Make a new scaler, and scale the three columns in `value_features` with
`.fit_transform`, as in the chapter's section "Feature scaling". Put the result
in a DataFrame named `X_value`, with the three column names, as in the chapter
for `X`.

Use a new `StandardScaler()` for this step. The chapter's `scaler` was fitted on the
six behavior features.

In [ ]:
value_features = ["revenue_12m", "orders_12m", "tenure_months"]

# TODO: scale the three value features and put them in a DataFrame
X_value = None

# --- Look: every scaled column has mean 0 and standard deviation 1 ---
X_value.describe().round(2).loc[["mean", "std"]]

## Step 2: Compute the inertia and the silhouette score for k from 2 to 7

Follow the loop in the chapter's section "Number of segments". For
each k in `range(2, 8)`, fit `KMeans(n_clusters=k, n_init=10, random_state=0)` on
`X_value`, and append one row: k, the model's `.inertia_`, and
`silhouette_score(X_value, m.labels_)`. Collect the rows in a DataFrame named
`kdf_value`.

In [ ]:
# TODO: one row per k: [k, inertia, silhouette]
rows = []

# --- Look: inertia and silhouette score for each k ---
kdf_value = pd.DataFrame(rows, columns=["k", "inertia", "silhouette"])
kdf_value.round(3)

## Step 3: Silhouette score at k = 4

In this lab k is fixed at 4, so that the value segments can be compared with the
four behavioral segments. The table in Step 2 is reported for reference. From it,
report two numbers: the silhouette score at k = 4, and the k with the highest
silhouette score in the table, with that score.

---

**Silhouette score at k = 4:** *replace with your answer*

**Highest silhouette score in the table:** *replace with your answer*

---

## Step 4: Fit the value segments and profile them

Fit `KMeans(n_clusters=4, n_init=10, random_state=0)` on `X_value`.
Store each customer's group number in a new column, `snap["value_segment"]`, with
`.fit_predict`, as in the chapter for `segment`. Then build the profile in the
original units with `.groupby`, as in the chapter's section "Segment profiles":
the mean of the three value features and of `churned` for each value
segment, plus a `customers` column from `value_counts()`.

In [ ]:
# TODO: fit the value segments and profile them in the original units
km_value = None
value_profile = None

# --- Look: the overall means, then the profile of each value segment ---
print(snap[value_features].mean().round(1))
value_profile

## Step 5: Profile patterns

For each value segment, compare its mean revenue, mean orders, and mean tenure with
the overall means printed above Step 4's profile. Write each comparison as "above" or
"below", in the order revenue, orders, tenure, and quote the segment's means. For
example, a segment with every mean above the overall mean is "above, above, above".

---

**Profile patterns:** *replace with your answer*

---

## Step 6: Compare the value segments with the behavioral segments

Make a crosstab of `snap["value_segment"]` against `snap["segment_name"]`, as in the
chapter's comparison of the three-segment and four-segment groupings. Each cell
holds the number of customers in one value segment and one behavioral segment.

In [ ]:
# TODO: crosstab of the value segments against the behavioral segments

## Step 7: Largest overlap for each behavioral segment

Read down each column of the crosstab. For each behavioral segment, report the value
segment that holds most of its customers, the number of those customers out of the
segment's total, and that share as a percentage.

---

**Largest overlap:** *replace with your answer*

---

## Step 8 (Evaluate lab): Compute the silhouette scores

Compute `silhouette_score(X_value, snap["value_segment"])`. Then compute the
silhouette score of the business types on the same three scaled features,
`silhouette_score(X_value, snap["business_type"])`, as in the chapter's evaluation.
Print both.

Compare silhouette scores only on the same features. The chapter's 0.296 was
computed on the six behavior features. It cannot be compared with these two scores.

In [ ]:
# TODO: silhouette score of the value segments and of the business types on X_value

## Step 9: Check the stability on ten subsamples

Follow the chapter's stability loop. For `r` in `range(10)`, draw 80% of the rows of
`X_value` with `X_value.sample(frac=0.8, random_state=r)`, fit
`KMeans(n_clusters=4, n_init=10, random_state=0)` on the sample, and assign all 872 customers with
`m.predict(X_value)`. Append the ARI between `snap["value_segment"]` and those
assignments, and print the smallest and the largest of the ten.

In [ ]:
# TODO: ARI between the value segments and refits on ten 80% subsamples
aris = []

## Step 10: Compare churn and revenue across the two groupings

Compute the mean of `churned` and of `revenue_12m` for each value segment, and the
same for each behavioral segment (`segment_name`), with `.groupby`, as in the
chapter's usefulness check. Print the two tables one after the other.

In [ ]:
# TODO: churn rate and mean revenue by value segment and by behavioral segment

## Step 11: Evaluation results and the grouping for the retention team

First, report three results: whether the silhouette score of the value segments is
higher or lower than that of the business types (Step 8), with both scores; the
smallest and the largest ARI from Step 9; and the spread in churn rates of each
grouping (Step 10), computed as the highest churn rate minus the lowest, in percentage
points. Second, name the grouping with the larger spread in churn rates. As in the
chapter's usefulness check, this grouping is given to the retention team.

---

**Evaluation results:** *replace with your answer*

**Grouping for the retention team:** *replace with your answer*

---

## Submission

Run **Restart and run all** and confirm the notebook executes top to bottom with no
errors. Then download it as `.ipynb` and upload it to the Lab 12 dropbox.

If you used an AI assistant, add a one-line note naming the tool and what it
generated, per the course policy in Appendix D.